In [1]:
import pandas as pd
import numpy as np
from dico_villes import dico_villes
from fuel_consumption_calc import calculer_fuel_feat

In [2]:
file_path_flights = "./data/Flights_20191201_20191231.csv"

# Chargement avec paramètres spécifiques
df_Flights = pd.read_csv(
    file_path_flights,
    sep=",",  # Définit le séparateur (souvent ; en France)
    encoding="utf-8",  # Encodage (essayer 'latin1' ou 'cp1252' si erreur)
    index_col=0,  # Utilise la 1ère colonne comme index (optionnel)
    na_values=["n/a", "?"],  # Convertit ces valeurs spécifiques en NaN (vide)
)

feat_model_df = pd.read_csv("./data/ac_model_coefficients.csv")

file_path_ac = "./data/ac_model_coefficients_airliners.csv"

In [3]:
blacklist_operators = [
    # --- AC Inconnu ---
    "ZZZ",
    # --- CARGO (Fret) ---
    "FDX",
    "UPS",
    "BCS",
    "TAY",
    "SRR",
    "NPT",
    "SWN",
    "MNB",
    "AIZ",
    "SRN",
    "ABR",
    "SWT",
    # --- JETS PRIVÉS & AFFAIRES ---
    "NJE",
    "VJT",
    "AHO",
    "GAC",
    "SVW",
    "MMD",
    "DOP",
    # --- HÉLICOPTÈRES ---
    "BHL",
    # --- HORS PÉRIMÈTRE (Israël, etc.) ---
    "ELY",
    "ISR",
    # --- CHARTER/WET LEASE COMPLEXE (Optionnel, à retirer pour pureté) ---
    "HKS",
    "AWC",
    "MSA",
]

In [4]:
def select_top_villles(df_v, n=10):
    list_top = np.arange(1, n + 1, 1)
    mask_villes = df_v["Rang"].isin(list_top)

    top_v = df_v[mask_villes]
    # print("Premiere villes : \n", top_v.head())
    return top_v


def select_flights_train(df_v, df_F):

    airports_france = df_v["ICAO"].unique()

    vols_France_mask = df_F["ADEP"].isin(airports_france) & df_F["ADES"].isin(
        airports_france
    )

    vols_France = df_F[vols_France_mask]

    vols_France = vols_France[vols_France["ICAO"] != "ZZZ"]

    # print(f"Nombre de vols trouvés : {len(vols_France)}")
    # print(vols_France[["ADEP", "ADES"]].head())

    return vols_France


def select_flights_europe(df):
    codes_europe = ["L", "E", "B"]

    # 3. On crée un filtre : Le 1er caractère de ADEP *ET* de ADES doit être dans la liste
    # .str[0] permet de prendre juste la première lettre de la colonne
    filtre_europe = df["ADEP"].str[0].isin(codes_europe) & df["ADES"].str[0].isin(
        codes_europe
    )

    # 4. On applique le filtre
    vols_europe = df[filtre_europe]

    print(f"Nombre de vols europe : {len(vols_europe)}")

    return vols_europe


def select_top_companies(df, n):
    df_filtered = df[~df["AC Operator"].isin(blacklist_operators)]
    classement = df_filtered["AC Operator"].value_counts().reset_index()
    top_n = classement.head(n)

    return top_n


def filter_planes_and_companies(df, df_ac, df_comp):
    all_ac = df_ac["ac_code_icao"].values
    all_companies = df_comp["AC Operator"].values

    df_filtered_ac = df[df["AC Type"].isin(all_ac)]
    df_filtered = df_filtered_ac[df_filtered_ac["AC Operator"].isin(all_companies)]

    print(f"Nombre de vols europe filtrés : {len(df_filtered)}")

    return df_filtered


def all_liaisons(df, seuil_vols=60):
    df_temp = df.copy()

    routes_triees = np.sort(df_temp[["ADEP", "ADES"]].values, axis=1)

    df_temp["Aeroport_1"] = routes_triees[:, 0]
    df_temp["Aeroport_2"] = routes_triees[:, 1]

    classement_liaisons = df_temp.groupby(["Aeroport_1", "Aeroport_2"]).size()

    classement_liaisons = classement_liaisons.sort_values(ascending=False)
    classement_liaisons = classement_liaisons.reset_index(name="Nombre_de_vols")

    classement_liaisons = classement_liaisons[
        classement_liaisons["Nombre_de_vols"] >= seuil_vols
    ]

    # print(f"Ensemble des liaisons (Sens confondus) :")
    # print(classement_liaisons.head())

    return classement_liaisons


def generate_dico(df_v):

    df_airports = df_villes[df_villes["ICAO"] != "NO_AIRPORT"]
    dict_villes_icao = df_airports.set_index("ICAO")["Nom"].to_dict()

    return dict_villes_icao


def name_liaison(df_liaisons, dico):

    df_liaisons_copy = df_liaisons.copy()

    df_liaisons_copy["Ville_1"] = df_liaisons_copy["Aeroport_1"].map(dico)
    df_liaisons_copy["Ville_2"] = df_liaisons_copy["Aeroport_2"].map(dico)

    df_liaisons_copy = df_liaisons_copy.dropna(subset=["Ville_1", "Ville_2"])

    # print(
    #     df_liaisons_copy[
    #         ["Aeroport_1", "Ville_1", "Aeroport_2", "Ville_2", "Nombre_de_vols"]
    #     ]
    # )

    return df_liaisons_copy


def calculate_emmissions(df_flights, df_feat_model):

    df_flights_copy = df_flights.copy()

    merged_df = df_flights_copy.merge(
        df_feat_model[
            [
                "ac_code_icao",
                "reduced_fuel_a1",
                "reduced_fuel_a2",
                "reduced_fuel_intercept",
            ]
        ],
        left_on="AC Type",
        right_on="ac_code_icao",
        how="left",
    )

    merged_df["Actual Distance Flown (km)"] = (
        merged_df["Actual Distance Flown (nm)"] * 1.852
    )
    df_flights_copy["Fuel_Emissions"] = (
        merged_df["reduced_fuel_a1"] * (merged_df["Actual Distance Flown (km)"] ** 2)
        + merged_df["reduced_fuel_a2"] * merged_df["Actual Distance Flown (km)"]
        + merged_df["reduced_fuel_intercept"]
    ).values

    return df_flights_copy


def mean_flight(df_flights):
    """
    Agrège les vols pour obtenir une moyenne des émissions et du temps
    par liaison (Aéroport A -> Aéroport B).
    """
    df = df_flights.copy()

    df["OFF_BLOCK"] = pd.to_datetime(
        df["ACTUAL OFF BLOCK TIME"], format="%d-%m-%Y %H:%M:%S", errors="coerce"
    )
    df["ARRIVAL"] = pd.to_datetime(
        df["ACTUAL ARRIVAL TIME"], format="%d-%m-%Y %H:%M:%S", errors="coerce"
    )

    df["Flight_Time_min"] = (df["ARRIVAL"] - df["OFF_BLOCK"]).dt.total_seconds() / 60.0

    # # Nettoyage des éventuels temps négatifs ou aberrants
    # df = df[df["Flight_Time_min"] > 0]

    edges_df = (
        df.groupby(["ADEP", "ADES"])
        .agg(
            mean_emissions=("Fuel_Emissions", "mean"),
            mean_time=("Flight_Time_min", "mean"),
            num_flights=("Fuel_Emissions", "count"),
        )
        .reset_index()
    )

    return edges_df


def filter_mean(edges_df, dico, all_liaisons):
    """
    Filtre les liaisons orientées (A -> B) issues de mean_flight
    selon un nombre minimum de vols, sans mélanger l'aller et le retour.
    """
    df_temp = edges_df.copy()
    copy_all_liasons = all_liaisons.copy()

    copy_all_liasons_aller = copy_all_liasons.rename(
        columns={"Aeroport_1": "ADEP", "Aeroport_2": "ADES"}
    )
    copy_all_liasons_retour = copy_all_liasons.rename(
        columns={"Aeroport_1": "ADES", "Aeroport_2": "ADEP"}
    )

    all_aller = pd.merge(
        df_temp,
        copy_all_liasons_aller[["ADEP", "ADES"]],
        on=["ADES", "ADEP"],
        how="inner",
    )
    all_retour = pd.merge(
        df_temp,
        copy_all_liasons_retour[["ADEP", "ADES"]],
        on=["ADES", "ADEP"],
        how="inner",
    )

    liaisons_filtrees = pd.concat([all_aller, all_retour])

    liaisons_finales = liaisons_filtrees.sort_values(by="num_flights", ascending=False)

    liaisons_finales["Ville_DEP"] = liaisons_finales["ADEP"].map(dico)
    liaisons_finales["Ville_DES"] = liaisons_finales["ADES"].map(dico)

    return liaisons_finales


def get_vols_liaison(df, code_depart, code_arrivee, sens_confondus=False):
    """
    Sélectionne les vols entre deux aéroports.

    Paramètres:
    - df : Le DataFrame contenant les vols
    - code_depart : Code ICAO de l'aéroport 1 (ex: 'LFPG')
    - code_arrivee : Code ICAO de l'aéroport 2 (ex: 'LFMN')
    - sens_confondus : Si True, récupère A->B ET B->A (Aller-Retour)
    """
    condition_aller = (df["ADEP"] == code_depart) & (df["ADES"] == code_arrivee)

    if not sens_confondus:
        resultat = df[condition_aller]

    else:
        condition_retour = (df["ADEP"] == code_arrivee) & (df["ADES"] == code_depart)
        resultat = df[condition_aller | condition_retour]

    return resultat

In [5]:
df_europe = select_flights_europe(df_Flights)
top_companies = select_top_companies(df_europe, 50)
list_top_companies = top_companies["AC Operator"].values

Nombre de vols europe : 530504


In [6]:
print(list_top_companies)

['RYR' 'DLH' 'THY' 'AFR' 'EZY' 'SAS' 'KLM' 'WZZ' 'EWG' 'AZA' 'BAW' 'VLG'
 'PGT' 'SWR' 'BEE' 'LOT' 'FIN' 'TAP' 'WIF' 'AUA' 'NAX' 'AEA' 'IBE' 'BEL'
 'ANE' 'EIN' 'OAL' 'LOG' 'EZS' 'BTI' 'AEE' 'CFE' 'SHT' 'SXS' 'ROT' 'TRA'
 'ASL' 'STK' 'SEH' 'EXS' 'LGL' 'BMS' 'DLA' 'CTN' 'BRX' 'CSA' 'CCM' 'ISS'
 'DTR' 'TVF']


In [7]:
df_final_flights = filter_planes_and_companies(df_europe, feat_model_df, top_companies)
df_liaisons = all_liaisons(df_final_flights)

Nombre de vols europe filtrés : 411148


In [8]:
df_liaisons

,Aeroport_1,Aeroport_2,Nombre_de_vols
0,LTBJ,LTFJ,1476
1,LTAC,LTFJ,1399
2,LEBL,LEMD,1296
3,ENGM,ENVA,1223
4,EGLL,EHAM,1162
...,...,...,...
1788,EDDB,EGGD,60
1789,LFBO,LFST,60
1790,EGPE,EHAM,60
1791,EGCC,LIRA,60


In [9]:
df_liaisons_named = name_liaison(df_liaisons, dico_villes)

In [10]:
print("NOmbres de liaisons :", len(df_liaisons_named))
print(df_liaisons_named.head(30))

NOmbres de liaisons : 1793
   Aeroport_1 Aeroport_2  Nombre_de_vols     Ville_1            Ville_2
0        LTBJ       LTFJ            1476       Izmir           Istanbul
1        LTAC       LTFJ            1399      Ankara           Istanbul
2        LEBL       LEMD            1296   Barcelone             Madrid
3        ENGM       ENVA            1223        Oslo          Trondheim
4        EGLL       EHAM            1162     Londres          Amsterdam
5        LEBL       LEPA            1150   Barcelone  Palma de Majorque
6        ENBR       ENGM            1146      Bergen               Oslo
7        EDDH       EDDM            1138    Hambourg             Munich
8        LTAI       LTFJ            1122     Antalya           Istanbul
9        EGLL       EIDW            1119     Londres             Dublin
10       ENGM       ENZV            1088        Oslo          Stavanger
11       EDDL       EDDM            1072  Düsseldorf             Munich
12       EDDF       EDDT            1

In [11]:
flight1 = get_vols_liaison(df_final_flights, "LTBJ", "LTFJ")
flight2 = get_vols_liaison(df_final_flights, "LTFJ", "LTBJ")

In [12]:
print(len(flight1))
print(len(flight2))
print(len(flight1) + len(flight2))

734
742
1476


In [13]:
mon_depart = "EKCH"
mon_arrivee = "LKPR"

# Récupérer seulement l'aller
vols_aller = get_vols_liaison(
    df_final_flights, mon_depart, mon_arrivee, sens_confondus=False
)

# Récupérer l'aller-retour (La "Ligne" au sens ferroviaire)
vols_ligne_entiere = get_vols_liaison(
    df_final_flights, mon_depart, mon_arrivee, sens_confondus=True
)

# --- Vérification ---
print(f"Vols {mon_depart} -> {mon_arrivee} : {len(vols_aller)}")
print(f"Vols sur la ligne {mon_depart} <-> {mon_arrivee} : {len(vols_ligne_entiere)}")

# Afficher un aperçu
print(vols_ligne_entiere[["ADEP", "ADES", "AC Operator", "AC Type"]].head())

Vols EKCH -> LKPR : 67
Vols sur la ligne EKCH <-> LKPR : 133
           ADEP  ADES AC Operator AC Type
ECTRL ID                                 
236570355  EKCH  LKPR         CSA    B737
236573620  LKPR  EKCH         RYR    B738
236574321  LKPR  EKCH         CSA    B737
236576578  EKCH  LKPR         RYR    B738
236577903  EKCH  LKPR         CSA    B737


In [14]:
all_airports = (
    pd.concat([df_final_flights["ADEP"], df_final_flights["ADES"]])
).unique()


all_airports_names = pd.concat(
    [
        df_final_flights["ADEP"].map(dico_villes),
        df_final_flights["ADES"].map(dico_villes),
    ]
).unique()

print("Length all airports Codes :", len(all_airports), "\n")
print("All airports Codes : \n", all_airports)
print("\n \n")
print("Length all airports Names :", len(all_airports_names), "\n")
print("All airports Names : \n ", all_airports_names)

Length all airports Codes : 471 

All airports Codes : 
 ['EGSS' 'LTAI' 'LTAF' 'EDDK' 'LTAC' 'LATI' 'EDDV' 'LTBJ' 'LCLK' 'LTBA'
 'BKPR' 'LTFM' 'LROP' 'LTAU' 'LTFH' 'LTCG' 'LIMC' 'LTFG' 'LTCA' 'LTFJ'
 'LTCB' 'LTAT' 'LTAN' 'LLBG' 'LGAV' 'LTDA' 'EFOU' 'LRIA' 'LBSF' 'EFHK'
 'EETN' 'LRCL' 'LRSB' 'LUKK' 'LRCV' 'LRTR' 'EYVI' 'LGTS' 'BIKF' 'LGIR'
 'LOWW' 'LBWN' 'EPGD' 'LRBC' 'LWSK' 'LGSA' 'LGRP' 'EPKK' 'EPWR' 'EPWA'
 'LTFE' 'LEMD' 'LEMG' 'EKYT' 'LEVC' 'ESGG' 'EVRA' 'ELLX' 'LFBO' 'EKCH'
 'EBBR' 'EDDT' 'LICJ' 'LICC' 'LFML' 'LHBP' 'LPPT' 'ENCN' 'LFBD' 'LSGG'
 'EDDS' 'ENBR' 'LEBL' 'LFLL' 'EKBI' 'LKPR' 'LPPR' 'ENZV' 'LIRN' 'EDDN'
 'EDDH' 'ESSA' 'LFMT' 'LCPH' 'EDDW' 'EDDL' 'ENGM' 'LYBE' 'EDDM' 'EPRZ'
 'LPMA' 'LIPX' 'LIPE' 'LIRQ' 'ENTO' 'LOWG' 'EDDF' 'ESSL' 'LIRF' 'EYKA'
 'LEPA' 'LSZH' 'LFRB' 'LFRS' 'LHDC' 'LFBP' 'LIPZ' 'ENVA' 'EIDW' 'LIME'
 'LFSB' 'LICA' 'LIMF' 'LFPG' 'EPPO' 'LFPO' 'LDZA' 'LIBD' 'EPKT' 'LIEE'
 'LQSA' 'LIML' 'LIBR' 'EHAM' 'LEGR' 'LIRP' 'LIRA' 'EBCI' 'LFMN' 'LTBS'
 'EDDC' 'LEAL' 'LDDU

In [15]:
df_final_flights_emissions = calculate_emmissions(df_final_flights, feat_model_df)

In [16]:
print(calculer_fuel_feat(df_final_flights.iloc[0]))

Type of aircraft :  B738
Distance flown (km) :  2746.516
9797.577056113876


In [17]:
mean = mean_flight(df_final_flights_emissions)

In [18]:
filtred_mean = filter_mean(mean, dico_villes, df_liaisons)

In [19]:
filtred_mean

,ADEP,ADES,mean_emissions,mean_time,num_flights,Ville_DEP,Ville_DES
1652,LTFJ,LTBJ,2618.039751,59.928616,742,Istanbul,Izmir
1745,LTBJ,LTFJ,2682.141134,54.847230,734,Izmir,Istanbul
1711,LTAC,LTFJ,2571.554358,48.169105,704,Ankara,Istanbul
1642,LTFJ,LTAC,2471.774927,52.996930,695,Istanbul,Ankara
683,LEMD,LEBL,3102.965256,67.040615,650,Madrid,Barcelone
...,...,...,...,...,...,...,...
900,LFRN,EHAM,2110.986085,74.671839,29,Rennes,Amsterdam
447,EPKT,EHEH,3893.542233,105.129310,29,Katowice,Eindhoven
1023,ENAL,ENVA,1932.726511,43.845679,27,Alesund,Trondheim
766,EGPA,EGPB,247.855689,35.425000,26,Kirkwall,Sumburgh


In [20]:
df_liaisons_named[
    (df_liaisons_named.Aeroport_1 == "EBBR") & (df_liaisons_named.Aeroport_2 == "LFMN")
]

,Aeroport_1,Aeroport_2,Nombre_de_vols,Ville_1,Ville_2
1048,EBBR,LFMN,118,Bruxelles,Nice


In [21]:
print(sum(mean.num_flights))

411148


In [22]:
print(sum(filtred_mean.num_flights))

348142


In [23]:
print(sum(df_liaisons_named.Nombre_de_vols))

348142


In [24]:
len(get_vols_liaison(df_final_flights, "BIKF", "EDDF"))

14

In [25]:
get_vols_liaison(mean, "LEMD", "LFLL")

,ADEP,ADES,mean_emissions,mean_time,num_flights
5025,LEMD,LFLL,2632.576117,93.814972,59


In [26]:
get_vols_liaison(mean, "LFLL", "LKPR")

,ADEP,ADES,mean_emissions,mean_time,num_flights
5666,LFLL,LKPR,2624.231267,91.873118,31


In [27]:
get_vols_liaison(mean, "LEMD", "LKPR")

,ADEP,ADES,mean_emissions,mean_time,num_flights
5052,LEMD,LKPR,6925.457596,161.666135,94
